<a href="https://colab.research.google.com/github/santiagoleguizamon10/TP-labo/blob/fran_pando/ROL3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ANÁLISIS DE PROTAGONISMO MEDIANTE LA POSESIÓN DEL JUEGO

In [1]:
# 1. Importación de librerías esenciales
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
# 2. Ruta dataset y empezar a trabajar con el
url = 'https://raw.githubusercontent.com/santiagoleguizamon10/TP-labo/main/data/Clausura26_tp_labo.csv'
df = pd.read_csv(url)

print("Dimensiones del dataset:", df.shape)
df.head(3)

Dimensiones del dataset: (194, 21)


,Fecha,Local,Visitante,Resultado (L),Resultado (V),Posesión de balón (L),Posesión de balón (V),Goles esperados (xG) (L),Goles esperados (xG) (V),Tiros totales (L),...,Tiros al arco (L),Tiros al arco (V),Córners (L),Córners (V),Faltas (L),Faltas (V),Tarjetas amarillas (L),Tarjetas amarillas (V),Tarjetas rojas (L),Tarjetas rojas (V)
0,Fecha 1,Club Atlético Belgrano,Rosario Central,2,1,48%,52%,1.50,1.39,13,...,5,3,6,6,10,13,0.0,2.0,0,0
1,Fecha 1,Sarmiento,Argentinos Juniors,2,3,45%,55%,1.50,1.42,12,...,3,5,7,2,15,14,3.0,6.0,0,1
2,Fecha 1,Defensa y Justicia,Aldosivi,1,1,72%,28%,1.06,0.59,15,...,5,3,6,2,9,15,2.0,3.0,0,0


# **3) Análisis de Causalidad entre la Posesión de Balón y la Probabilidad de Victoria en Condición de Local**

¿El control de la posesión de la pelota está determinado por la condición de local o por la identidad táctica de cada equipo, y en qué medida esta tenencia condiciona el resultado final?

En este bloque se busca diferenciar si los equipos tienen más la pelota simplemente por jugar en su estadio o si responden a un estilo de juego propio, analizando cómo impacta ese estilo en la diferencia de goles.


**A) Limpieza y Transformación de Datos en Python**

Para este bloque se ejecutan tres transformaciones clave en la celda del notebook:

1. Conversión de Tipo de Dato: La columna Posesión de balón (L) se encuentra originalmente como texto (object) con símbolo de porcentaje (%). Se convierte a valor numérico flotante (float) mediante .str.rstrip('%').astype(float).
2. Creación de Variables Derivadas:
  

    * Dif_Goles: Diferencia neta de goles del encuentro (Resultado (L) - Resultado (V)).

    * Pts_Local: Asignación de puntos obtenidos por el local (3 si gana, 1 si empata, 0 si pierde).

3. Agrupación por Tramos de Posesión: Se segmenta la posesión del local en tres categorías estratégicas con pd.cut():
      *   Baja: < 45%;
      *   Media: 45% - 55%;
      *   Alta: > 55%.




In [3]:
# 1. Conversión de tipo de dato
df['Posesion_L_num'] = df['Posesión de balón (L)'].str.rstrip('%').astype(float)
df['Posesion_V_num'] = df['Posesión de balón (V)'].str.rstrip('%').astype(float)

# 2a. Diferencia de goles
df['Dif_Goles'] = df['Resultado (L)'] - df['Resultado (V)']

# 2b. Puntos asignados al local
def asignar_puntos_local(row):
    if row['Resultado (L)'] > row['Resultado (V)']:
        return 3
    elif row['Resultado (L)'] == row['Resultado (V)']:
        return 1
    else:
        return 0

df['Pts_Local'] = df.apply(asignar_puntos_local, axis=1)

# 3. Tramos de posesión
bins = [0, 45, 55, 100]
labels = ['Baja (<45%)', 'Media (45-55%)', 'Alta (>55%)']
df['Tramo_Posesion_L'] = pd.cut(df['Posesion_L_num'], bins=bins, labels=labels)

print("Transformaciones completadas con éxito.")

Transformaciones completadas con éxito.


**B) Análisis previo: Identidad de juego vs Factor localía**

Previo a la evaluación del impacto del control del balón sobre el resultado de los partidos, resulta necesario determinar si la tenencia constituye un reflejo de la condición de localía o si responde a la identidad táctica de cada equipo, permitiendo prevenir sesgos de atribución causal.


In [4]:
# [Celda Markdown en Colab]
# ### Paso 3: Análisis previo: Identidad Táctica vs. Factor Campo
# Evaluamos si la posesión está determinada por la localía (+2.1% en promedio general)
# o por el modelo de juego de cada institución.

# Agregación por club
locales = df.groupby('Local')['Posesion_L_num'].agg(['mean', 'count']).rename(columns={'mean': 'Pos_Local_Prom', 'count': 'PJ_L'})
visitantes = df.groupby('Visitante')['Posesion_V_num'].agg(['mean', 'count']).rename(columns={'mean': 'Pos_Visitante_Prom', 'count': 'PJ_V'})

# Unión de datos por equipo
equipos = locales.join(visitantes)
equipos['Pos_Global_Prom'] = (equipos['Pos_Local_Prom'] * equipos['PJ_L'] + equipos['Pos_Visitante_Prom'] * equipos['PJ_V']) / (equipos['PJ_L'] + equipos['PJ_V'])
equipos['Efecto_Localia_Pos'] = equipos['Pos_Local_Prom'] - equipos['Pos_Visitante_Prom']

# Top 5 mayor posesión y Top 5 menor posesión
equipos_ordenados = equipos.sort_values(by='Pos_Global_Prom', ascending=False)

print("--- TOP 5 EQUIPOS CON MAYOR TENENCIA GLOBAL ---")
print(equipos_ordenados[['Pos_Global_Prom', 'Pos_Local_Prom', 'Pos_Visitante_Prom']].head(5))

print("\n--- TOP 5 EQUIPOS CON MENOR TENENCIA GLOBAL (MODELO REACTIVO) ---")
print(equipos_ordenados[['Pos_Global_Prom', 'Pos_Local_Prom', 'Pos_Visitante_Prom']].tail(5))

--- TOP 5 EQUIPOS CON MAYOR TENENCIA GLOBAL ---
                         Pos_Global_Prom  Pos_Local_Prom  Pos_Visitante_Prom
Local                                                                       
River Plate                    66.000000       63.857143           68.500000
Argentinos Juniors             60.769231       63.400000           59.125000
Boca Juniors                   60.692308       58.833333           62.285714
Estudiantes de La Plata        59.461538       58.857143           60.166667
Rosario Central                58.153846       59.333333           57.142857

--- TOP 5 EQUIPOS CON MENOR TENENCIA GLOBAL (MODELO REACTIVO) ---
                   Pos_Global_Prom  Pos_Local_Prom  Pos_Visitante_Prom
Local                                                                 
Aldosivi                 43.769231       49.333333           39.000000
Atlético Tucumán         39.461538       39.714286           39.166667
Banfield                 38.923077       38.500000           3